# 02 決定木・ランダムフォレスト・ExtraTrees

## このノートの目的
- **決定木**がどうやって「分けて」予測するのかを、手で計算して理解する。
- 木を1本だけ使うと**過学習**しやすいこと、**たくさんの木を合わせる**（アンサンブル）と安定することを確かめる。
- **ランダムフォレスト**と **ExtraTrees** の違いを知り、**ハイパーパラメータ**（`min_samples_leaf`・`max_features`・`class_weight`）の効き方を見る。

## キーワード
| 用語 | やさしい説明 |
|---|---|
| **決定木** | 「湿度が◯以上か？」のような質問を繰り返して、標本を分けていくモデル。分けた先（**葉**）の多数派で予測する。 |
| **不純度（ジニ）** | 1つのグループに「あり」と「なし」がどれだけ混ざっているか。0 なら完全に純粋。木は**不純度が最も減る分け方**を選ぶ。 |
| **過学習** | 学習データを**覚えすぎて**、新しいデータでは当たらなくなること。木を深くするほど起きやすい。 |
| **アンサンブル** | 多数のモデルの予測を平均して、安定させること。 |
| **バギング** | 学習データを**復元抽出**（重複を許して選ぶ）で何通りも作り、それぞれで木を育てる方法。 |
| **ランダムフォレスト** | バギング ＋ 各分岐で**ランダムに選んだ一部の特徴量だけ**を候補にする。木どうしが似すぎないようにする。 |
| **ExtraTrees** | 分岐の**しきい値そのものをランダムに**選ぶ（最良のしきい値を探さない）。さらに木がバラバラになり、速く、なめらかになる。 |
| **min_samples_leaf** | 葉に最低何標本残すか。**大きいほど木は浅く（単純に）なる**。 |
| **max_features** | 各分岐で試す特徴量の数。 |
| **class_weight** | 少数派（陽性）の標本の重み。`balanced` なら、陽性を重く扱う。 |

In [ ]:
# 準備: 必要なライブラリを読み込み、src フォルダの部品（説明変数を作る dataset.py など）を使えるようにします
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve() / "src"))   # ノートブックは notebooks/ にあるので、1つ上の src を探す
plt.rcParams["font.family"] = ["Hiragino Sans", "Arial Unicode MS", "sans-serif"]  # 日本語を表示するための設定
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 140)
import dataset as ds   # 説明変数と正解を作る部品（src/dataset.py）

In [ ]:
# 目的: 以降で使うデータを用意する。2010〜2019年を学習、2020〜2024年を評価にする簡単な分け方（学習用の簡易版。本番は leave-one-year-out）
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import average_precision_score

x = ds.build("naha")
f = ds.features("naha", change=False)
target = "y20_region"
dev = x[(x.year <= 2024) & x.wind_ok].reset_index(drop=True)          # 評価に使える年（2025年は最終評価用なので使わない）
tr = dev[dev.year <= 2019]; va = dev[dev.year >= 2020]
med = tr[f].median()                                                    # 欠測は学習側の中央値で埋める
Xtr, ytr, Xva, yva = tr[f].fillna(med), tr[target], va[f].fillna(med), va[target]
print(f"学習: {len(tr)}標本（陽性 {ytr.sum()}）／ 評価: {len(va)}標本（陽性 {yva.sum()}、陽性率 {yva.mean():.3f}）")

**出力の見方**: 評価側の陽性は**数十標本（十数日）**しかありません。以降の数字は**ぶれやすい**ので、傾向を見る目的で読んでください（正確な比較はノート03で扱います）。

## 1. 決定木 — 「分ける」ことの意味

まず、**深さ2だけ**の小さな木を作って、図にします。

In [ ]:
# 目的: 深さ2の決定木を作り、どう分けているかを図にする
small = ["sun_morning", "dewdep", "wind_mean", "humid"]                  # 見やすいように特徴量を4つに絞る
tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=0).fit(Xtr[small], ytr)
fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(tree, feature_names=small, class_names=["なし", "あり"], filled=True, impurity=True, ax=ax, fontsize=8)
plt.show()

**出力の見方**: 各四角が1つの**分岐**です。
- 1行目 `sun_morning <= ○○` が質問です。**True なら左、False なら右**に進みます。
- `gini` は**不純度**（0に近いほど純粋）、`samples` はそこに残った標本数、`value` は［なし, あり］の（重みづけした）数です。
- 色は多数派（青が「あり」寄り）。一番下の四角（**葉**）の多数派が、その標本の予測になります。

### 不純度を手で計算してみる

木は「**不純度が最も減る**分け方」を探しています。1つの特徴量について、自分で探してみましょう。
**ジニ不純度** = 1 − p² − (1−p)²（p は陽性の割合）。分けたあとは、2つのグループの不純度を**標本数で重みづけ**して足します。

In [ ]:
# 目的: 1つの特徴量（午前の日照時間 sun_morning）で、不純度が最も減るしきい値を、総当たりで探す
def gini(y):
    p = np.mean(y) if len(y) else 0.0
    return 1 - p ** 2 - (1 - p) ** 2

v = Xtr["sun_morning"].to_numpy(); yy = ytr.to_numpy()
cands = np.unique(np.quantile(v, np.linspace(0.02, 0.98, 97)))             # しきい値の候補
gain = []
for t in cands:
    left, right = yy[v <= t], yy[v > t]
    after = (len(left) * gini(left) + len(right) * gini(right)) / len(yy)   # 分けたあとの不純度（重みづけ平均）
    gain.append(gini(yy) - after)                                           # 減った量
best = cands[int(np.argmax(gain))]
print(f"分ける前の不純度: {gini(yy):.4f}")
print(f"自分で探した最良のしきい値: sun_morning <= {best:.2f}（不純度の減少 {max(gain):.5f}）")

stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(Xtr[["sun_morning"]], ytr)   # 深さ1（切り株）。重みなし
print(f"sklearn の木が選んだしきい値: sun_morning <= {stump.tree_.threshold[0]:.2f}")
plt.figure(figsize=(6, 3)); plt.plot(cands, gain); plt.axvline(best, color="r", ls="--")
plt.xlabel("しきい値（sun_morning）"); plt.ylabel("不純度の減少"); plt.show()

**出力の見方**: 自分で探したしきい値（例 0.10）と、sklearn の木のしきい値（例 0.15）は**見かけが違いますが、同じ分け方**です。
日照時間は 0.1 刻みの値なので、「0.10 以下」と「0.15 以下」は同じ標本を左に振り分けます（sklearn は、隣り合う値の**中間**をしきい値にします）。
決定木は、**この計算をすべての特徴量・すべての分岐で繰り返している**だけです。

## 2. 過学習 — 木を深くすると何が起きるか

木を**深く**すると、学習データには完璧に当たるようになります。しかし、**新しいデータ**ではどうでしょう。

In [ ]:
# 目的: 木の深さを変えて、学習データと評価データの PR-AUC を比べる
rows = []
for d in range(1, 21):
    m = DecisionTreeClassifier(max_depth=d, class_weight="balanced", random_state=0).fit(Xtr, ytr)
    rows.append({"深さ": d, "学習データ": average_precision_score(ytr, m.predict_proba(Xtr)[:, 1]),
                 "評価データ": average_precision_score(yva, m.predict_proba(Xva)[:, 1])})
r = pd.DataFrame(rows).set_index("深さ")
r.plot(marker="o", figsize=(7, 3.5)); plt.axhline(yva.mean(), color="gray", ls="--"); plt.ylabel("PR-AUC（灰色の破線は評価側の陽性率）"); plt.show()
r.round(3).T

**出力の見方**: 深さを増やすと、**学習データの PR-AUC はどんどん上がり（1に近づく）**ますが、**評価データはほとんど上がらないか、下がります**。
学習データと評価データの差が開くのが**過学習**です。1本の木は、深くすると「覚えるだけ」になります。

## 3. たくさんの木を合わせる — ランダムフォレストと ExtraTrees

1本の木は不安定でも、**多数の木を平均**すると安定します。ただし、木が**どれも似ていたら**意味がありません。
そこで、木をバラバラにする工夫をします。

| 手法 | 工夫 |
|---|---|
| **ランダムフォレスト** | ① 学習データを**ブートストラップ**（復元抽出）で作り直す ② 各分岐で、**ランダムに選んだ一部の特徴量だけ**を候補にする。しきい値は**最良のものを探す**。 |
| **ExtraTrees** | 各分岐で、候補の特徴量それぞれに**ランダムなしきい値**を引き、その中で最良のものを選ぶ。学習データは**そのまま**（ブートストラップなし）。 |

In [ ]:
# 目的: 木1本・ランダムフォレスト・ExtraTrees を、シードを変えて5回ずつ評価する（乱数で結果が少し変わるため）
def eva(make, seeds=range(5)):
    ap = []
    for s in seeds:
        m = make(s).fit(Xtr, ytr)
        ap.append(average_precision_score(yva, m.predict_proba(Xva)[:, 1]))
    return np.mean(ap), np.std(ap)

models = {
    "木1本（深さ制限なし）": lambda s: DecisionTreeClassifier(class_weight="balanced", random_state=s),
    "木1本（min_samples_leaf=20）": lambda s: DecisionTreeClassifier(min_samples_leaf=20, class_weight="balanced", random_state=s),
    "ランダムフォレスト": lambda s: RandomForestClassifier(n_estimators=200, min_samples_leaf=20, class_weight="balanced_subsample", n_jobs=-1, random_state=s),
    "ExtraTrees": lambda s: ExtraTreesClassifier(n_estimators=200, min_samples_leaf=20, class_weight="balanced_subsample", n_jobs=-1, random_state=s),
}
res = {k: eva(v) for k, v in models.items()}
pd.DataFrame(res, index=["PR-AUC（平均）", "標準偏差（シード間）"]).T.round(3)

**出力の見方**: **木1本**は、`min_samples_leaf` を大きくして浅くしても、PR-AUC が約 0.04〜0.06 で、**フォレスト系（約 0.10）の半分程度**です。多数の木を平均する効果が見えます。
ランダムフォレストと ExtraTrees は**近い値**です。このプロジェクトの結論（ノート03）でも、**両者の差は偶然の範囲**でした。

> 📝 標準偏差は**乱数（シード）によるぶれ**だけです。評価データ自体が少ないことによる不確かさは、これよりずっと大きい（→ ノート03）。

## 4. ハイパーパラメータを変えてみる

**ハイパーパラメータ**は、学習の前に人が決める設定です。1つずつ変えて、効き方を見ます。

In [ ]:
# 目的: ExtraTrees の min_samples_leaf を変えて、評価データの PR-AUC を見る
leafs = [1, 3, 5, 10, 20, 50, 100]
r1 = [eva(lambda s, L=L: ExtraTreesClassifier(n_estimators=200, min_samples_leaf=L, class_weight="balanced_subsample", n_jobs=-1, random_state=s), range(3)) for L in leafs]
r1 = pd.DataFrame(r1, index=leafs, columns=["平均", "標準偏差"])
ax = r1["平均"].plot(marker="o", yerr=r1["標準偏差"], capsize=3, figsize=(6, 3)); ax.set_xscale("log")
ax.axhline(yva.mean(), color="gray", ls="--"); ax.set_xlabel("min_samples_leaf（大きいほど木は浅い）"); ax.set_ylabel("PR-AUC"); plt.show()
r1.round(3).T

**出力の見方**: `min_samples_leaf=1`（木を限界まで深く）より、**ある程度大きくした方が良い**傾向です。大きくしすぎると、今度は**単純すぎて**下がります（**過少適合**）。
ちょうどよい値は**データによって違い**、評価データが少ないと見つけにくいです。

In [ ]:
# 目的: max_features（各分岐で試す特徴量の数）と class_weight（陽性の重み）を変えて比べる
rows = []
for mf in ["sqrt", 0.3, 0.6, 1.0]:
    for cw in [None, "balanced_subsample"]:
        a, sd = eva(lambda s: ExtraTreesClassifier(n_estimators=200, min_samples_leaf=20, max_features=mf, class_weight=cw, n_jobs=-1, random_state=s), range(3))
        rows.append({"max_features": mf, "class_weight": cw or "なし", "PR-AUC": a, "標準偏差": sd})
pd.DataFrame(rows).pivot(index="max_features", columns="class_weight", values="PR-AUC").round(3)

**出力の見方**
- **max_features**: 小さいほど木がバラバラになります（ランダムネスが強い）。大きいと、どの木も似てきます。
- **class_weight**: 陽性が少ないとき、`balanced` 系にすると**陽性を見逃しにくく**なります。ただし、**予測確率の大きさがずれる**副作用があります（→ ノート06）。
  PR-AUC は**順位**で決まるので、重みの有無による差は小さいことが多く、この実験でも**差は小さい**（重みなしのほうが少し高い組み合わせもある）です。

## まとめ
- 決定木は「**不純度が最も減る分け方**」を繰り返す。1本は**過学習**しやすい。
- **多数の木の平均**（ランダムフォレスト・ExtraTrees）は安定する。両者の差は、このデータでは小さい。
- **min_samples_leaf を大きくして、木を浅く**するのが効く。ただし、評価データが少ないと、最適値は見えにくい。

## 演習
1. **1.** の `max_depth=2` を `max_depth=3` に変えて図を見てください（観察点: 葉の数はいくつに増えるか。葉の `value` はどう変わるか）。
2. **1.** の最良しきい値の探索で、`sun_morning` を `dewdep`（気温−露点）に変えてください（観察点: 不純度の減少量は `sun_morning` より大きいか、小さいか）。
3. **3.** の `n_estimators=200` を `20` に変えてください（観察点: 標準偏差は大きくなるか。木の数が少ないと、シードによるぶれが増える）。
4. **4.** の `leafs` に `200` を加えてください（観察点: 単純すぎて PR-AUC が下がり始めるか）。